## CVE & CWE Mapping dataset
This file will pull the dataset from https://huggingface.co/datasets/zefang-liu/cve-and-cwe-mapping-dataset/ and use the data gathered through the scrapers to make it more complete and remove any noise from the final dataset.

### Requirements

In [ ]:
!pip install pandas
!pip install -q huggingface_hub python-dotenv


## Gather data

In [64]:
import pandas as pd
import json
import re

This is the original dataset we're going to modify

In [65]:
df = pd.read_csv("hf://datasets/zefang-liu/cve-and-cwe-mapping-dataset/Global_Dataset.csv")
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,DESCRIPTION,CWE-ID
0,1,CVE-1999-0001,NaN,5.0,MEDIUM,ip_input.c in BSD-derived TCP/IP implementatio...,CWE-20
1,2,CVE-1999-0002,NaN,10.0,HIGH,Buffer overflow in NFS mountd gives root acces...,CWE-119
2,3,CVE-1999-0003,NaN,10.0,HIGH,Execute commands as root via buffer overflow i...,NVD-CWE-Other
3,4,CVE-1999-0004,NaN,5.0,MEDIUM,"MIME buffer overflow in email clients, e.g. So...",NVD-CWE-Other
4,5,CVE-1999-0005,NaN,10.0,HIGH,Arbitrary command execution via IMAP buffer ov...,NVD-CWE-Other


This is the data where we scraped from CWE's websites

In [66]:
# == KAGGLE/COLAB ==

# !git clone https://github.com/regularpooria/CWE_CVE_Mapping # Kaggle
# BASE_DIR = "./CWE_CVE_Mapping" # Kaggle

# == LOCAL ==
BASE_DIR = "../" # Local
with open(f"{BASE_DIR}/scrapers/scraped_data/CWE_SOFTWARE.json", "r", encoding="utf-8") as f:
    CWE = json.load(f)
    
list_of_CWEs = set()
for category_id, category in CWE.items():
    list_of_CWEs.add(category_id)
    list_of_CWEs |= set(list(category["children"].keys()))
    

## Merging the 2 data sources

In [67]:
# Removing any ambiguous cases.

values_to_remove = ['NVD-CWE-Other', 'NVD-CWE-noinfo']
df = df[~df['CWE-ID'].isin(values_to_remove)]
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,DESCRIPTION,CWE-ID
0,1,CVE-1999-0001,NaN,5.0,MEDIUM,ip_input.c in BSD-derived TCP/IP implementatio...,CWE-20
1,2,CVE-1999-0002,NaN,10.0,HIGH,Buffer overflow in NFS mountd gives root acces...,CWE-119
6,7,CVE-1999-0007,NaN,5.0,MEDIUM,Information from SSL-encrypted sessions via PK...,CWE-327
25,26,CVE-1999-0027,NaN,7.2,HIGH,root privileges via buffer overflow in eject c...,CWE-119
176,177,CVE-1999-0179,NaN,5.0,MEDIUM,Windows NT crashes or locks up when a Samba cl...,CWE-17


In [68]:
# Extract the year
def extract_year(CVE_ID):
    return int(CVE_ID.split("-")[1])

df["YEAR"] = df["CVE-ID"].apply(extract_year)
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,DESCRIPTION,CWE-ID,YEAR
0,1,CVE-1999-0001,NaN,5.0,MEDIUM,ip_input.c in BSD-derived TCP/IP implementatio...,CWE-20,1999
1,2,CVE-1999-0002,NaN,10.0,HIGH,Buffer overflow in NFS mountd gives root acces...,CWE-119,1999
6,7,CVE-1999-0007,NaN,5.0,MEDIUM,Information from SSL-encrypted sessions via PK...,CWE-327,1999
25,26,CVE-1999-0027,NaN,7.2,HIGH,root privileges via buffer overflow in eject c...,CWE-119,1999
176,177,CVE-1999-0179,NaN,5.0,MEDIUM,Windows NT crashes or locks up when a Samba cl...,CWE-17,1999


In [69]:
# Check to see if they are in CWE699 (SOFTWARE)
def is_in_CWE699(CWE_ID):
    id = CWE_ID.split("-")[-1]
    return id in list_of_CWEs

df["IS-IN-CWE699"] = df["CWE-ID"].apply(is_in_CWE699)
df = df[~df['IS-IN-CWE699'] == False]
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,DESCRIPTION,CWE-ID,YEAR,IS-IN-CWE699
195,196,CVE-1999-0199,9.8,7.5,CRITICAL,manual/search.texi in the GNU C Library (aka g...,CWE-252,1999,True
222,223,CVE-1999-0226,NaN,10.0,HIGH,Windows NT TCP/IP processes fragmented IP pack...,CWE-19,1999,True
279,280,CVE-1999-0284,NaN,7.5,HIGH,Denial of service to NT mail servers including...,CWE-120,1999,True
380,381,CVE-1999-0385,NaN,10.0,HIGH,The LDAP bind function in Exchange 5.5 has a b...,CWE-120,1999,True
382,383,CVE-1999-0387,NaN,7.8,HIGH,A legacy credential caching mechanism used in ...,CWE-255,1999,True


In [70]:
df = df.rename(columns={"DESCRIPTION": "CVE-DESCRIPTION"})
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,CVE-DESCRIPTION,CWE-ID,YEAR,IS-IN-CWE699
195,196,CVE-1999-0199,9.8,7.5,CRITICAL,manual/search.texi in the GNU C Library (aka g...,CWE-252,1999,True
222,223,CVE-1999-0226,NaN,10.0,HIGH,Windows NT TCP/IP processes fragmented IP pack...,CWE-19,1999,True
279,280,CVE-1999-0284,NaN,7.5,HIGH,Denial of service to NT mail servers including...,CWE-120,1999,True
380,381,CVE-1999-0385,NaN,10.0,HIGH,The LDAP bind function in Exchange 5.5 has a b...,CWE-120,1999,True
382,383,CVE-1999-0387,NaN,7.8,HIGH,A legacy credential caching mechanism used in ...,CWE-255,1999,True


In [71]:
# Children to parent mapping

children_to_parent = {}
for category_id, category in CWE.items():
    for child_id, child in category["children"].items():
        child_id = str(child_id)
        item_to_add = str(category_id)
        item_to_add = (f"CWE-{category_id}", category["summary"])
        if child_id in children_to_parent:
            children_to_parent[child_id]["parents"].append(item_to_add)  
        else:
            children_to_parent[child_id] = child
            children_to_parent[child_id]["parents"] = [item_to_add]


In [72]:
def get_parent_id(CWE_ID):
    id = CWE_ID.split("-")[-1] # "CWE-199" -> "199"
    if id not in children_to_parent:
        return pd.NA
    
    return children_to_parent[id]["parents"]

df["PARENTS"] = df["CWE-ID"].apply(get_parent_id)
df = df[~df["PARENTS"].isna()]
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,CVE-DESCRIPTION,CWE-ID,YEAR,IS-IN-CWE699,PARENTS
195,196,CVE-1999-0199,9.8,7.5,CRITICAL,manual/search.texi in the GNU C Library (aka g...,CWE-252,1999,True,"[(CWE-389, This category includes weaknesses t..."
279,280,CVE-1999-0284,NaN,7.5,HIGH,Denial of service to NT mail servers including...,CWE-120,1999,True,"[(CWE-1218, Weaknesses in this category are re..."
380,381,CVE-1999-0385,NaN,10.0,HIGH,The LDAP bind function in Exchange 5.5 has a b...,CWE-120,1999,True,"[(CWE-1218, Weaknesses in this category are re..."
485,486,CVE-1999-0491,NaN,4.6,MEDIUM,The prompt parsing in bash allows a local user...,CWE-94,1999,True,"[(CWE-137, Weaknesses in this category are rel..."
502,503,CVE-1999-0509,NaN,10.0,HIGH,"Perl, sh, csh, or other shell interpreters are...",CWE-94,1999,True,"[(CWE-137, Weaknesses in this category are rel..."


In [84]:
def get_CWE_description(CWE_ID):
    id = CWE_ID.split("-")[-1] # "CWE-199" -> "199"
    return children_to_parent[id]["description"]

df["CWE-DESCRIPTION"] = df["CWE-ID"].apply(get_CWE_description)
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,CVE-DESCRIPTION,CWE-ID,YEAR,IS-IN-CWE699,PARENTS,CWE-DESCRIPTION,CVE-DESCRIPTION-NORMALIZED
195,196,CVE-1999-0199,9.8,7.5,CRITICAL,manual/search.texi in the GNU C Library (aka g...,CWE-252,1999,True,"[(CWE-389, This category includes weaknesses t...",The product does not check the return value fr...,manual/search.texi in the C Library before lac...
279,280,CVE-1999-0284,NaN,7.5,HIGH,Denial of service to NT mail servers including...,CWE-120,1999,True,"[(CWE-1218, Weaknesses in this category are re...",The product copies an input buffer to an outpu...,"Denial of service to mail servers including,, ..."
380,381,CVE-1999-0385,NaN,10.0,HIGH,The LDAP bind function in Exchange 5.5 has a b...,CWE-120,1999,True,"[(CWE-1218, Weaknesses in this category are re...",The product copies an input buffer to an outpu...,The bind function in has a buffer overflow tha...
485,486,CVE-1999-0491,NaN,4.6,MEDIUM,The prompt parsing in bash allows a local user...,CWE-94,1999,True,"[(CWE-137, Weaknesses in this category are rel...",The product constructs all or part of a code s...,The prompt parsing in bash allows a local user...
502,503,CVE-1999-0509,NaN,10.0,HIGH,"Perl, sh, csh, or other shell interpreters are...",CWE-94,1999,True,"[(CWE-137, Weaknesses in this category are rel...",The product constructs all or part of a code s...,"Perl, sh, csh, or other shell interpreters are..."


## Post processing on the data

In [ ]:
from utils import remove_capital_words, remove_parentheses

In [97]:
tmp = "Apache Log4j2 2.0-beta9 through 2.12.1 and 2.13.0 through 2.15.0 JNDI (user patch is recommended) features used in configuration, log messages, and parameters do not protect against attacker controlled LDAP and other JNDI related endpoints. An attacker who can control log messages or log message parameters can execute arbitrary code loaded from LDAP servers when message lookup substitution is enabled. From log4j 2.15.0, this behavior has been disabled by default. From version 2.16.0, this functionality has been completely removed. Note that this vulnerability is specific to log4j-core and does not affect log4net, log4cxx, or other Apache Logging Services projects"
print(tmp)
print(remove_capital_words(remove_parentheses(tmp)))

Apache Log4j2 2.0-beta9 through 2.12.1 and 2.13.0 through 2.15.0 JNDI (user patch is recommended) features used in configuration, log messages, and parameters do not protect against attacker controlled LDAP and other JNDI related endpoints. An attacker who can control log messages or log message parameters can execute arbitrary code loaded from LDAP servers when message lookup substitution is enabled. From log4j 2.15.0, this behavior has been disabled by default. From version 2.16.0, this functionality has been completely removed. Note that this vulnerability is specific to log4j-core and does not affect log4net, log4cxx, or other Apache Logging Services projects
Apache through and through JNDI features used in configuration, log messages, and parameters do not protect against attacker controlled LDAP and other JNDI related endpoints. An attacker who can control log messages or log message parameters can execute arbitrary code loaded from LDAP servers when message lookup substitution i

In [94]:
removed_words

[('Ipswitch',
  'Denial of service to NT mail servers including Ipswitch, Mdaemon, and Exchange through a buffer overflow in the SMTP HELO command'),
 ('Mdaemon',
  'Denial of service to NT mail servers including Ipswitch, Mdaemon, and Exchange through a buffer overflow in the SMTP HELO command'),
 ('Exchange',
  'Denial of service to NT mail servers including Ipswitch, Mdaemon, and Exchange through a buffer overflow in the SMTP HELO command'),
 ('Exchange',
  'The LDAP bind function in Exchange 5.5 has a buffer overflow that allows a remote attacker to conduct a denial of service or execute commands'),
 ('Explorer',
  'Internet Explorer 5.0 and 5.01 allows remote attackers to modify or execute files via the Import/Export Favorites feature, aka the "ImportExportFavorites" vulnerability'),
 ('ImportExportFavorites',
  'Internet Explorer 5.0 and 5.01 allows remote attackers to modify or execute files via the Import/Export Favorites feature, aka the "ImportExportFavorites" vulnerability')

In [95]:
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,CVE-DESCRIPTION,CWE-ID,YEAR,IS-IN-CWE699,PARENTS,CWE-DESCRIPTION,CVE-DESCRIPTION-NORMALIZED
195,196,CVE-1999-0199,9.8,7.5,CRITICAL,manual/search.texi in the GNU C Library (aka g...,CWE-252,1999,True,"[(CWE-389, This category includes weaknesses t...",The product does not check the return value fr...,manual/search.texi in the GNU C Library before...
279,280,CVE-1999-0284,NaN,7.5,HIGH,Denial of service to NT mail servers including...,CWE-120,1999,True,"[(CWE-1218, Weaknesses in this category are re...",The product copies an input buffer to an outpu...,Denial of service to NT mail servers including...
380,381,CVE-1999-0385,NaN,10.0,HIGH,The LDAP bind function in Exchange 5.5 has a b...,CWE-120,1999,True,"[(CWE-1218, Weaknesses in this category are re...",The product copies an input buffer to an outpu...,The LDAP bind function in has a buffer overflo...
485,486,CVE-1999-0491,NaN,4.6,MEDIUM,The prompt parsing in bash allows a local user...,CWE-94,1999,True,"[(CWE-137, Weaknesses in this category are rel...",The product constructs all or part of a code s...,The prompt parsing in bash allows a local user...
502,503,CVE-1999-0509,NaN,10.0,HIGH,"Perl, sh, csh, or other shell interpreters are...",CWE-94,1999,True,"[(CWE-137, Weaknesses in this category are rel...",The product constructs all or part of a code s...,"Perl, sh, csh, or other shell interpreters are..."


In [98]:
# Save the final dataset locally and commit it to the Hugging Face dataset repository.
import os
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import HfApi

load_dotenv()
HF_REPO_ID = os.environ.get(
    "HF_DATASET_REPO_ID",
    "regularpooria/CVE_CWE_Software_Mapping_Dataset",
)

UPLOAD_DIR = Path("hf_upload")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = UPLOAD_DIR / "Global_Dataset.csv"
df.to_csv(DATASET_PATH, index=False)

api = HfApi(token=os.environ.get("HF_TOKEN"))
commit_url = api.upload_folder(
    folder_path=str(UPLOAD_DIR),
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    commit_message="Update CVE-CWE Software Weakness Mapping Dataset",
)

print(f"Saved {len(df):,} rows to {DATASET_PATH}")
print(f"Committed dataset and dataset card to {commit_url}")

Saved 60,897 rows to hf_upload/Global_Dataset.csv
Committed dataset and dataset card to https://huggingface.co/datasets/regularpooria/CVE_CWE_Software_Mapping_Dataset/commit/82f0569034f6fecefa901178f26a7014bab33760
